# LRModelQ2

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

In [2]:
# --- Load Clean Data (Yelp 3-class sentiment) ---
df = pd.read_csv(DATA_FILE)

In [3]:
# --- Split into train/test (stratify by 'sentiment', random state for reproducibility) ---
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

In [4]:
# --- Build Logistic Regression pipeline ---
# tfidf: text -> numeric features. Shared config across all 4 Q2 models for a fair
#        comparison: 10k vocab, 1-2 grams, min_df=2, sublinear_tf, strip_accents.
# clf: multinomial logistic regression, balanced classes, more iters to converge
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                              min_df=2, sublinear_tf=True, strip_accents="unicode")),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced',
                               n_jobs=-1, random_state=42))
])

In [5]:
# --- Train, save for future use, predict based on test data ---
pipeline.fit(X_train, y_train)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / 'lr_pipeline.joblib')
preds = pipeline.predict(X_test)

C:\Users\yingx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [6]:
# --- Report base performance measures ---
print("=== Q2: Logistic Regression Classification Report (Yelp 3-class sentiment) ===")
print(classification_report(y_test, preds))

=== Q2: Logistic Regression Classification Report (Yelp 3-class sentiment) ===
              precision    recall  f1-score   support

    negative       0.85      0.80      0.82      2400
     neutral       0.47      0.59      0.53      1200
    positive       0.85      0.80      0.83      2400

    accuracy                           0.76      6000
   macro avg       0.73      0.73      0.73      6000
weighted avg       0.78      0.76      0.77      6000

